In [ ]:
# =============================================================================
# Deep Reinforcement Learning for Structural Damage Detection
# Algorithm: Deep Deterministic Policy Gradient (DDPG)
# =============================================================================
#
# Research Context:
# This notebook implements a continuous-control reinforcement learning
# framework for structural damage quantification in a multi-story shear
# building model.
#
# Objective:
# The agent receives structural response data (accelerations) and predicts
# story-wise damage coefficients.
#
# State (Observation):
#   Flattened normalized acceleration time-series
#
# Action:
#   Continuous damage vector for each story (bounded between 0.5 and 1.0)
#
# Reward:
#   Negative Mean Absolute Error between predicted and true damage
#
# This implementation serves as the RL benchmark to be compared against
# classical NExT-PCA sensitivity-based damage detection.
# =============================================================================



# =============================================================================
# 1. Installation (Colab Only)
# =============================================================================
!pip install stable-baselines3[extra] shimmy scikit-learn --quiet



# =============================================================================
# 2. Imports
# =============================================================================
import os
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gymnasium as gym
from gymnasium import spaces

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from stable_baselines3 import DDPG
from stable_baselines3.common.noise import NormalActionNoise



# =============================================================================
# 3. Dataset Upload and Extraction
# =============================================================================
from google.colab import files

uploaded = files.upload()

with zipfile.ZipFile("RL_Dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/RL_Dataset")

dataset_dir = "/content/RL_Dataset"

print("Dataset extracted successfully.")



# =============================================================================
# 4. Load Labels
# =============================================================================
with open(os.path.join(dataset_dir, "labels.json"), "r") as f:
    labels_data = json.load(f)

scaler = MinMaxScaler(feature_range=(-1, 1))

print(f"Loaded {len(labels_data)} labeled scenarios.")



# =============================================================================
# 5. Custom Reinforcement Learning Environment
# =============================================================================
class DamageDetectionEnv(gym.Env):
    """
    Custom environment for structural damage detection.
    """

    def __init__(self, dataset_dir, labels):

        super().__init__()

        self.dataset_dir = dataset_dir
        self.labels = labels

        self.num_floors = 5
        self.num_samples = 100

        # Observation space
        self.observation_space = spaces.Box(
            low=-1,
            high=1,
            shape=(self.num_floors * self.num_samples,),
            dtype=np.float32
        )

        # Continuous action space (damage ratios)
        self.action_space = spaces.Box(
            low=0.5,
            high=1.0,
            shape=(self.num_floors,),
            dtype=np.float32
        )

    def reset(self, seed=None, options=None):

        idx = np.random.randint(0, len(self.labels))
        label_info = self.labels[idx]

        file_path = os.path.join(
            self.dataset_dir,
            f"{label_info['scenario']}.csv"
        )

        data = pd.read_csv(file_path).values[:self.num_samples, :]

        scaled_data = scaler.fit_transform(data)

        self.current_observation = scaled_data.flatten().astype(np.float32)
        self.current_label = np.array(label_info["damage_pattern"])

        return self.current_observation, {}

    def step(self, action):

        error = np.abs(action - self.current_label)

        reward = -np.mean(error)

        terminated = True
        truncated = False

        info = {
            "true_damage": self.current_label.tolist(),
            "predicted": action.tolist()
        }

        return self.current_observation, reward, terminated, truncated, info



# =============================================================================
# 6. Environment Initialization
# =============================================================================
env = DamageDetectionEnv(dataset_dir=dataset_dir, labels=labels_data)

n_actions = env.action_space.shape[-1]

action_noise = NormalActionNoise(
    mean=np.zeros(n_actions),
    sigma=0.05 * np.ones(n_actions)
)



# =============================================================================
# 7. Model Definition and Training
# =============================================================================
model = DDPG(
    "MlpPolicy",
    env,
    action_noise=action_noise,
    verbose=1
)

model.learn(total_timesteps=20000)

print("Training completed.")



# =============================================================================
# 8. Save and Reload Model
# =============================================================================
model.save("ddpg_damage_detector")

model = DDPG.load("ddpg_damage_detector")

print("Model saved and reloaded successfully.")



# =============================================================================
# 9. Evaluation Phase
# =============================================================================
results = []

for _ in range(20):

    obs, _ = env.reset()

    action, _ = model.predict(obs, deterministic=True)

    _, _, _, _, info = env.step(action)

    results.append(info)



# =============================================================================
# 10. Error Analysis
# =============================================================================
true_vals = np.array([r["true_damage"] for r in results])
pred_vals = np.array([r["predicted"] for r in results])

errors = np.abs(true_vals - pred_vals)

plt.figure(figsize=(10, 5))
plt.boxplot(errors, labels=[f"Floor {i+1}" for i in range(5)])
plt.title("Absolute Prediction Error per Floor (DDPG)")
plt.ylabel("|Predicted - True|")
plt.grid(True)
plt.show()



# =============================================================================
# 11. Quantitative Metrics
# =============================================================================
y_true = true_vals.flatten()
y_pred = pred_vals.flatten()

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
accuracy = np.mean(np.abs(y_true - y_pred) < 0.05) * 100

print("DDPG Performance Metrics")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"Accuracy (|error| < 0.05): {accuracy:.2f}%")
